In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import xarray as xr
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Reshape, UpSampling2D, Layer, ZeroPadding2D, Cropping2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from google.colab import drive
import shutil

class Sampling(Layer):
    """Uses (z_mean, z_log_var) to sample z."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

class VAE(tf.keras.Model):

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)

        reconstructed = self.decoder(z)

        return reconstructed

    def __init__(self, encoder, decoder, beta = 0.0, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def test_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)

        reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.square(data - reconstruction), axis=[1, 2]))

        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))

        total_loss = reconstruction_loss + (self.beta * kl_loss)

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.square(data - reconstruction), axis=[1, 2]))


            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))

            total_loss = reconstruction_loss + self.beta * kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }


In [ ]:
LATENT_DIM = 16
LATENT_DIM_NP = 128


def build_BK_vae():
    # --- ENCODER ---
    input_img = Input(shape=(24, 72, 1))
    x = Conv2D(32, (3,3), strides=(2,2), activation='relu', padding='same')(input_img)
    x = Conv2D(64, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dense(128, activation='relu')(x)

    z_mean = Dense(LATENT_DIM, name='z_mean')(x)
    z_log_var = Dense(LATENT_DIM, name='z_log_var')(x)
    z = Sampling()([z_mean, z_log_var])

    encoder = Model(input_img, [z_mean, z_log_var, z], name='bk_encoder')

    # --- DECODER ---
    latent_inputs = Input(shape=(LATENT_DIM,))
    x = Dense(128, activation='relu')(latent_inputs)
    x = Dense(512, activation='relu')(x)
    x = Dense(6 * 18 * 128, activation='relu')(x)
    x = Reshape((6, 18, 128))(x)

    x = UpSampling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(32, (3,3), activation='relu', padding='same')(x)
    decoded = Conv2D(1, (3,3), activation='linear', padding='same')(x)

    decoder = Model(latent_inputs, decoded, name='bk_decoder')
    return encoder, decoder


def build_NAO_vae():
    # --- ENCODER ---
    input_img = Input(shape=(40, 40, 1))
    x = Conv2D(32, (3, 3), strides=(2, 2), activation='relu', padding='same')(input_img)
    x = Conv2D(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
    x = Conv2D(128, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dense(128, activation='relu')(x)

    z_mean = Dense(LATENT_DIM, name='z_mean')(x)
    z_log_var = Dense(LATENT_DIM, name='z_log_var')(x)
    z = Sampling()([z_mean, z_log_var])

    encoder = Model(input_img, [z_mean, z_log_var, z], name='nao_encoder')

    # --- DECODER ---
    latent_inputs = Input(shape=(LATENT_DIM,))
    x = Dense(128, activation='relu')(latent_inputs)
    x = Dense(512, activation='relu')(x)
    x = Dense(5 * 5 * 256, activation='relu')(x)
    x = Reshape((5, 5, 256))(x)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    decoded = Conv2D(1, (3, 3), activation='linear', padding='same')(x)

    decoder = Model(latent_inputs, decoded, name='nao_decoder')
    return encoder, decoder


def build_NP_vae():
    # --- ENCODER ---
    input_img = Input(shape=(41, 512, 1))
    x = ZeroPadding2D(padding=((4, 3), (0, 0)))(input_img)
    x = Conv2D(32, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Conv2D(64, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Conv2D(128, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Conv2D(256, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Flatten()(x)
    x = Dense(2048, activation='relu')(x)
    x = Dense(512, activation='relu')(x)

    z_mean = Dense(LATENT_DIM_NP, name='z_mean')(x)
    z_log_var = Dense(LATENT_DIM_NP, name='z_log_var')(x)
    z = Sampling()([z_mean, z_log_var])

    encoder = Model(input_img, [z_mean, z_log_var, z], name='np_encoder')

    # --- DECODER ---
    latent_inputs = Input(shape=(LATENT_DIM_NP,))
    x = Dense(512, activation='relu')(latent_inputs)
    x = Dense(2048, activation='relu')(x)
    x = Dense(3 * 32 * 256, activation='relu')(x)
    x = Reshape((3, 32, 256))(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = UpSampling2D((2,2))(x)
    decoded_padded = Conv2D(1, (3,3), activation='linear', padding='same', dtype='float32')(x)
    decoded = Cropping2D(cropping=((4, 3), (0, 0)), dtype='float32')(decoded_padded)



    decoder = Model(latent_inputs, decoded, name='np_decoder')



    return encoder, decoder

def build_NP_SIC_vae():
    # --- ENCODER ---
    input_img = Input(shape=(32, 360, 1))

    x = Conv2D(32, (3,3), strides=(2,2), activation='relu', padding='same')(input_img)
    x = Conv2D(64, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Conv2D(128, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Conv2D(256, (3,3), activation='relu', padding='same')(x)
    x = Flatten()(x)
    x = Dense(2048, activation='relu')(x)
    x = Dense(512, activation='relu')(x)

    z_mean = Dense(LATENT_DIM_NP, name='z_mean')(x)
    z_log_var = Dense(LATENT_DIM_NP, name='z_log_var')(x)
    z = Sampling()([z_mean, z_log_var])

    encoder = Model(input_img, [z_mean, z_log_var, z], name='np_sic_encoder')

    # --- DECODER ---
    latent_inputs = Input(shape=(LATENT_DIM_NP,))
    x = Dense(512, activation='relu')(latent_inputs)
    x = Dense(2048, activation='relu')(x)
    x = Dense(4 * 45 * 256, activation='relu')(x)
    x = Reshape((4, 45, 256))(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(32, (3,3), activation='relu', padding='same')(x)

    decoded = Conv2D(1, (3,3), activation='linear', padding='same', dtype='float32')(x)

    decoder = Model(latent_inputs, decoded, name='np_sic_decoder')
    return encoder, decoder


def build_BK_SIC_vae():
    # --- ENCODER ---
    input_img = Input(shape=(16, 64, 1))

    x = Conv2D(32, (3,3), strides=(2,2), activation='relu', padding='same')(input_img)
    x = Conv2D(64, (3,3), strides=(2,2), activation='relu', padding='same')(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dense(128, activation='relu')(x)

    z_mean = Dense(LATENT_DIM, name='z_mean')(x)
    z_log_var = Dense(LATENT_DIM, name='z_log_var')(x)
    z = Sampling()([z_mean, z_log_var])

    encoder = Model(input_img, [z_mean, z_log_var, z], name='bk_sic_encoder')

    # --- DECODER ---
    latent_inputs = Input(shape=(LATENT_DIM,))
    x = Dense(128, activation='relu')(latent_inputs)
    x = Dense(512, activation='relu')(x)
    x = Dense(4 * 16 * 128, activation='relu')(x)
    x = Reshape((4, 16, 128))(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = UpSampling2D((2,2))(x)
    x = Conv2D(32, (3,3), activation='relu', padding='same')(x)

    decoded = Conv2D(1, (3,3), activation='linear', padding='same', dtype='float32')(x)

    decoder = Model(latent_inputs, decoded, name='bk_sic_decoder')
    return encoder, decoder

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/python/Masters data'

file_config = {
    "BK_TAS_full.nc": build_BK_vae,
    "BK_hfls_full.nc": build_BK_vae,
    "BK_zg_full.nc": build_BK_vae,
    "NAO_hfls_full.nc": build_NAO_vae,
    "NAO_TAS_full.nc": build_NAO_vae,
    "NAO_zg_full.nc": build_NAO_vae,
    "NP_hfls_full.nc": build_NP_vae,
    "NP_TAS_full.nc": build_NP_vae,
    "NP_zg_full.nc": build_NP_vae,
    "NP_SIC_full.nc": build_NP_SIC_vae,
    "BK_SIC_full.nc": build_BK_SIC_vae
}

def load_and_clean_data(filepath, memmap_path):
    ds = xr.open_dataset(filepath, engine="h5netcdf", chunks="auto")

    if "TAS" in filepath:
        combined = ds['tas_lr'].fillna(ds['tas_sr']).dropna(dim="lat", how="any").rename("tas")
    elif "hfls" in filepath:
        combined = ds['hfls_lr'].fillna(ds['hfls_sr']).dropna(dim="lat", how="any").rename("hfls")
    elif "zg" in filepath:
        da_lr = ds['zg_lr'].rename({'member_lr': 'member'})
        da_sr = ds['zg_sr'].rename({'member_sr': 'member'})
        zg_combined = xr.concat([da_lr, da_sr], dim='member')
        combined = zg_combined.squeeze('plev').dropna(dim="lat", how="any").rename("hfls")
        del da_lr, da_sr, zg_combined
    elif "NP_SIC" in filepath:
        combined = ds['__xarray_dataarray_variable__'].dropna(dim="lat", how="any").rename("sic")
        combined = combined.pad(lat=(0, 2), constant_values=-1.0)

    elif "BK_SIC" in filepath:
        combined = ds['__xarray_dataarray_variable__'].dropna(dim="lat", how="any").rename("sic")
        combined = combined.pad(lon=(0, 13), constant_values=-1.0)
    else:
        var_name = [v for v in ds.data_vars if v not in ['lat', 'lon', 'time', 'member']][0]
        combined = ds[var_name].dropna(dim="lat", how="any")

    combined_stacked = combined.stack(samples=("member", "time")).transpose("samples", "lat", "lon")

    members = combined_stacked['member'].values
    times = combined_stacked['time'].values

    total_samples = len(members)
    lat_dim = combined_stacked.shape[1]
    lon_dim = combined_stacked.shape[2]
    memmap_shape = (total_samples, lat_dim, lon_dim, 1)

    print(f"Allocating {memmap_shape} memory-map on disk...")
    X_memmap = np.memmap(memmap_path, dtype='float32', mode='w+', shape=memmap_shape)

    chunk_size = 2000
    for i in range(0, total_samples, chunk_size):
        end = min(i + chunk_size, total_samples)
        chunk = combined_stacked.isel(samples=slice(i, end)).values.astype(np.float32)
        X_memmap[i:end, :, :, 0] = chunk

    X_memmap.flush()

    del ds, combined, combined_stacked
    if 'chunk' in locals(): del chunk
    gc.collect()

    X_train = np.memmap(memmap_path, dtype='float32', mode='r', shape=memmap_shape)

    return X_train, members, times

# --- Execution ---
for filename, arch_builder in file_config.items():
    print(f"\n{'='*60}\nProcessing: {filename}\n{'='*60}")

    # input_path = os.path.join(DRIVE_PATH, filename)
    input_path_drive = os.path.join(DRIVE_PATH, filename)
    output_path = os.path.join(DRIVE_PATH, filename.replace(".nc", "_latent_ae.csv"))
    checkpoint_path = os.path.join(DRIVE_PATH, filename.replace(".nc", "_checkpoint_ae.weights.h5"))

    local_temp_path = f"/content/{filename}"

    if os.path.exists(output_path):
        print(f"Skipping {filename} - Output CSV already exists.")
        continue

    try:
        print("Copying 5GB file from Drive to local Colab storage...")
        shutil.copy(input_path_drive, local_temp_path)

        local_memmap_path = f"/content/{filename.replace('.nc', '_memmap.dat')}"

        print("Extracting data via xarray chunking...")
        X_train, members, times = load_and_clean_data(local_temp_path, local_memmap_path)

        split_idx = int(len(X_train) * 0.8)
        X_train_split = X_train[:split_idx]
        X_val_split = X_train[split_idx:]

        def make_memmap_gen(memmap_arr):
            def _gen():
                for i in range(len(memmap_arr)):
                    yield (memmap_arr[i], memmap_arr[i])
            return _gen

        def make_predict_gen(memmap_arr):
            def _gen():
                for i in range(len(memmap_arr)):
                    yield memmap_arr[i]
            return _gen

        lat_dim = X_train.shape[1]
        lon_dim = X_train.shape[2]
        spatial_shape = (lat_dim, lon_dim, 1)

        output_sig = (
            tf.TensorSpec(shape=spatial_shape, dtype=tf.float32),
            tf.TensorSpec(shape=spatial_shape, dtype=tf.float32)
        )

        with tf.device('/CPU:0'):
            train_ds = tf.data.Dataset.from_generator(
                make_memmap_gen(X_train_split), output_signature=output_sig
            ).batch(64).prefetch(tf.data.AUTOTUNE)

            val_ds = tf.data.Dataset.from_generator(
                make_memmap_gen(X_val_split), output_signature=output_sig
            ).batch(64).prefetch(tf.data.AUTOTUNE)

        print("Building architecture...")
        encoder, decoder = arch_builder()
        vae = VAE(encoder, decoder)
        dynamic_shape = (None, lat_dim, lon_dim, 1)
        vae.build(dynamic_shape)
        vae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3))

        if os.path.exists(checkpoint_path):
            print(f"Found saved weights at {checkpoint_path}. Loading them now...")
            vae.load_weights(checkpoint_path)

            print("Setting learning rate to 1e-3 for resumed training...")
            vae.optimizer.learning_rate.assign(2.5e-4)

        else:
            print("No saved weights found. Starting fresh.")

        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)
        checkpoint = ModelCheckpoint(filepath=checkpoint_path, monitor='val_loss', save_best_only=True, save_weights_only=True)

        print("Starting training...")
        vae.fit(train_ds, validation_data=val_ds, epochs=100, callbacks=[early_stopping, reduce_lr, checkpoint])

        print("Extracting latent space...")
        with tf.device('/CPU:0'):
            predict_ds = tf.data.Dataset.from_generator(
                make_predict_gen(X_train),
                output_signature=tf.TensorSpec(shape=spatial_shape, dtype=tf.float32)
            ).batch(64).prefetch(tf.data.AUTOTUNE)

        z_mean, _, _ = vae.encoder.predict(predict_ds)

        print("Saving to CSV...")
        df = pd.DataFrame({'member': members, 'time': times})

        current_latent_dim = z_mean.shape[1]
        latent_col_names = [f'latent_{filename.split("_")[1]}_{i}' for i in range(current_latent_dim)]

        latent_df = pd.DataFrame(z_mean, columns=latent_col_names)
        final_df = pd.concat([df, latent_df], axis=1)

        final_df.to_csv(output_path, index=False)
        print(f"Successfully saved {output_path}")

    except Exception as e:
        print(f"FAILED on {filename}: {e}")

    finally:
        # --- CLEANUP ---
        if os.path.exists(local_temp_path):
            os.remove(local_temp_path)
        if os.path.exists(local_memmap_path):
            os.remove(local_memmap_path)

        if 'predict_ds' in locals(): del predict_ds
        if 'train_ds' in locals(): del train_ds, val_ds
        if 'X_train_da' in locals(): del X_train_da
        if 'vae' in locals(): del vae, encoder, decoder
        if 'z_mean' in locals(): del z_mean
        if 'final_df' in locals(): del df, latent_df, final_df

        tf.keras.backend.clear_session()
        gc.collect()


Processing: BK_TAS_full.nc
Skipping BK_TAS_full.nc - Output CSV already exists.

Processing: BK_hfls_full.nc
Skipping BK_hfls_full.nc - Output CSV already exists.

Processing: BK_zg_full.nc
Skipping BK_zg_full.nc - Output CSV already exists.

Processing: NAO_hfls_full.nc
Skipping NAO_hfls_full.nc - Output CSV already exists.

Processing: NAO_TAS_full.nc
Skipping NAO_TAS_full.nc - Output CSV already exists.

Processing: NAO_zg_full.nc
Skipping NAO_zg_full.nc - Output CSV already exists.

Processing: NP_hfls_full.nc
Skipping NP_hfls_full.nc - Output CSV already exists.

Processing: NP_TAS_full.nc
Copying 5GB file from Drive to local Colab storage...
Extracting data via xarray chunking...
Allocating (89604, 41, 512, 1) memory-map on disk...
Building architecture...
Found saved weights at /content/drive/MyDrive/python/Masters data/NP_TAS_full_checkpoint_ae.weights.h5. Loading them now...
Setting learning rate to 1e-3 for resumed training...
Starting training...
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 62 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


   1121/Unknown 100s 64ms/step - kl_loss: 1751.0400 - loss: 2.7991 - reconstruction_loss: 2.7991

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1121/1121 ━━━━━━━━━━━━━━━━━━━━ 123s 85ms/step - kl_loss: 1734.5691 - loss: 2.7610 - reconstruction_loss: 2.7610 - val_kl_loss: 1718.2701 - val_loss: 2.8612 - val_reconstruction_loss: 2.8612 - learning_rate: 2.5000e-04
Epoch 2/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 70s 62ms/step - kl_loss: 1747.0811 - loss: 2.5926 - reconstruction_loss: 2.5926 - val_kl_loss: 1736.3448 - val_loss: 2.7875 - val_reconstruction_loss: 2.7875 - learning_rate: 2.5000e-04
Epoch 3/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 68s 61ms/step - kl_loss: 1769.5150 - loss: 2.5367 - reconstruction_loss: 2.5367 - val_kl_loss: 1765.6385 - val_loss: 2.6995 - val_reconstruction_loss: 2.6995 - learning_rate: 2.5000e-04
Epoch 4/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 65s 58ms/step - kl_loss: 1788.3464 - loss: 2.5299 - reconstruction_loss: 2.5299 - val_kl_loss: 1775.5573 - val_loss: 2.8451 - val_reconstruction_loss: 2.8451 - learning_rate: 2.5000e-04
Epoch 5/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 64s 57ms/step - kl_loss: 1795.7371 - loss: 2.5091 - 

...

1121/1121 ━━━━━━━━━━━━━━━━━━━━ 86s 68ms/step - kl_loss: 233.8753 - loss: 139.6898 - reconstruction_loss: 139.6898 - val_kl_loss: 344.7791 - val_loss: 38.1158 - val_reconstruction_loss: 38.1158 - learning_rate: 0.0010
Epoch 2/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 81s 72ms/step - kl_loss: 309.5461 - loss: 42.6045 - reconstruction_loss: 42.6045 - val_kl_loss: 377.6538 - val_loss: 37.6989 - val_reconstruction_loss: 37.6989 - learning_rate: 0.0010
Epoch 3/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 69s 61ms/step - kl_loss: 544.8259 - loss: 22.6097 - reconstruction_loss: 22.6097 - val_kl_loss: 660.2571 - val_loss: 14.7161 - val_reconstruction_loss: 14.7161 - learning_rate: 0.0010
Epoch 4/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 70s 63ms/step - kl_loss: 641.3407 - loss: 14.3310 - reconstruction_loss: 14.3310 - val_kl_loss: 615.1606 - val_loss: 14.1625 - val_reconstruction_loss: 14.1625 - learning_rate: 0.0010
Epoch 5/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 70s 62ms/step - kl_loss: 655.6216 - loss: 11.3071 - reconst

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1121/1121 ━━━━━━━━━━━━━━━━━━━━ 128s 97ms/step - kl_loss: 72.3676 - loss: 692.5785 - reconstruction_loss: 692.5785 - val_kl_loss: 126.2340 - val_loss: 267.1033 - val_reconstruction_loss: 267.1033 - learning_rate: 0.0010
Epoch 2/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 109s 97ms/step - kl_loss: 124.1262 - loss: 183.9469 - reconstruction_loss: 183.9469 - val_kl_loss: 167.4636 - val_loss: 121.4822 - val_reconstruction_loss: 121.4822 - learning_rate: 0.0010
Epoch 3/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 93s 83ms/step - kl_loss: 190.7348 - loss: 113.2427 - reconstruction_loss: 113.2427 - val_kl_loss: 202.3575 - val_loss: 98.7225 - val_reconstruction_loss: 98.7225 - learning_rate: 0.0010
Epoch 4/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 142s 126ms/step - kl_loss: 220.9232 - loss: 94.7452 - reconstruction_loss: 94.7452 - val_kl_loss: 268.2624 - val_loss: 75.8322 - val_reconstruction_loss: 75.8322 - learning_rate: 0.0010
Epoch 5/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 137s 122ms/step - kl_loss: 290.4405 - loss: 75.9

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1121/1121 ━━━━━━━━━━━━━━━━━━━━ 53s 32ms/step - kl_loss: 25.3842 - loss: 33.9023 - reconstruction_loss: 33.9023 - val_kl_loss: 45.5946 - val_loss: 6.7609 - val_reconstruction_loss: 6.7609 - learning_rate: 0.0010
Epoch 2/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 26s 23ms/step - kl_loss: 53.5384 - loss: 7.6667 - reconstruction_loss: 7.6667 - val_kl_loss: 70.7213 - val_loss: 3.6074 - val_reconstruction_loss: 3.6074 - learning_rate: 0.0010
Epoch 3/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - kl_loss: 63.7961 - loss: 4.8511 - reconstruction_loss: 4.8511 - val_kl_loss: 69.3914 - val_loss: 3.0605 - val_reconstruction_loss: 3.0605 - learning_rate: 0.0010
Epoch 4/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 26s 23ms/step - kl_loss: 68.2167 - loss: 4.0651 - reconstruction_loss: 4.0651 - val_kl_loss: 75.6424 - val_loss: 2.6986 - val_reconstruction_loss: 2.6986 - learning_rate: 0.0010
Epoch 5/100
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - kl_loss: 76.2357 - loss: 3.1657 - reconstruction_loss: 3.1657 - val

In [ ]:
# AE
# BK TAS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - kl_loss: 512.7732 - loss: 0.0637 - reconstruction_loss: 0.0637 - val_kl_loss: 515.0189 - val_loss: 0.0652 - val_reconstruction_loss: 0.0652 - learning_rate: 2.5000e-04
# BK HFLS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 24s 22ms/step - kl_loss: 383.7237 - loss: 0.1791 - reconstruction_loss: 0.1791 - val_kl_loss: 386.1635 - val_loss: 0.2166 - val_reconstruction_loss: 0.2166 - learning_rate: 1.2500e-04
# BK ZG
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 24s 21ms/step - kl_loss: 460.7874 - loss: 0.0066 - reconstruction_loss: 0.0066 - val_kl_loss: 462.9591 - val_loss: 0.0069 - val_reconstruction_loss: 0.0069 - learning_rate: 1.5625e-05
# BK SIC
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 79s 71ms/step - kl_loss: 1938.4133 - loss: 2.1518 - reconstruction_loss: 2.1518 - val_kl_loss: 1922.7368 - val_loss: 2.2968 - val_reconstruction_loss: 2.2968 - learning_rate: 1.2500e-04


# NAO HFLS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - kl_loss: 274.0308 - loss: 0.0807 - reconstruction_loss: 0.0807 - val_kl_loss: 274.6498 - val_loss: 0.0931 - val_reconstruction_loss: 0.0931 - learning_rate: 1.0000e-05
# NAO TAS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - kl_loss: 411.9453 - loss: 0.0223 - reconstruction_loss: 0.0223 - val_kl_loss: 409.3086 - val_loss: 0.0233 - val_reconstruction_loss: 0.0233 - learning_rate: 1.2500e-04
# NAO ZG
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - kl_loss: 1021.9778 - loss: 0.0082 - reconstruction_loss: 0.0082 - val_kl_loss: 1017.3010 - val_loss: 0.0083 - val_reconstruction_loss: 0.0083 - learning_rate: 1.0000e-05

# NP HFLS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 68s 61ms/step - kl_loss: 1663.1605 - loss: 2.9938 - reconstruction_loss: 2.9938 - val_kl_loss: 1668.3196 - val_loss: 4.1867 - val_reconstruction_loss: 4.1867 - learning_rate: 1.2500e-04
# NP TAS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 68s 61ms/step - kl_loss: 2027.2024 - loss: 1.7438 - reconstruction_loss: 1.7438 - val_kl_loss: 2015.2809 - val_loss: 1.8812 - val_reconstruction_loss: 1.8812 - learning_rate: 6.2500e-05
# NP ZG
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 68s 61ms/step - kl_loss: 3624.3542 - loss: 0.6267 - reconstruction_loss: 0.6267 - val_kl_loss: 3616.2441 - val_loss: 0.6460 - val_reconstruction_loss: 0.6460 - learning_rate: 1.5625e-05
# NP SIC
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 100s 89ms/step - kl_loss: 1454.6144 - loss: 11.8124 - reconstruction_loss: 11.8124 - val_kl_loss: 1454.7638 - val_loss: 16.5072 - val_reconstruction_loss: 16.5072 - learning_rate: 1.2500e-04




In [ ]:
# VAE + B
# BK TAS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - kl_loss: 3.1732 - loss: 2.6752 - reconstruction_loss: 1.0886 - val_kl_loss: 3.1655 - val_loss: 2.6591 - val_reconstruction_loss: 1.0764 - learning_rate: 3.1250e-05
# BK HFLS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 26s 23ms/step - kl_loss: 3.8123 - loss: 3.7378 - reconstruction_loss: 1.8316 - val_kl_loss: 3.7895 - val_loss: 3.7249 - val_reconstruction_loss: 1.8301 - learning_rate: 1.2500e-04
# BK ZG
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 26s 23ms/step - kl_loss: 3.0248 - loss: 2.8145 - reconstruction_loss: 1.3022 - val_kl_loss: 3.0347 - val_loss: 2.8214 - val_reconstruction_loss: 1.3041 - learning_rate: 1.5625e-05
# BK SIC
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 25s 23ms/step - kl_loss: 4.9215 - loss: 5.2598 - reconstruction_loss: 2.7990 - val_kl_loss: 4.9358 - val_loss: 5.3400 - val_reconstruction_loss: 2.8721 - learning_rate: 1.5625e-05

# NAO hfls
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - kl_loss: 1.8632 - loss: 2.3349 - reconstruction_loss: 1.4033 - val_kl_loss: 1.8756 - val_loss: 2.3389 - val_reconstruction_loss: 1.4011 - learning_rate: 6.2500e-05
# NAO TAS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 24s 22ms/step - kl_loss: 1.9986 - loss: 2.3058 - reconstruction_loss: 1.3065 - val_kl_loss: 2.0432 - val_loss: 2.1452 - val_reconstruction_loss: 1.1236 - learning_rate: 0.0010
# NAO ZG
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - kl_loss: 2.7359 - loss: 2.4583 - reconstruction_loss: 1.0903 - val_kl_loss: 2.7197 - val_loss: 2.4483 - val_reconstruction_loss: 1.0885 - learning_rate: 6.2500e-05

# NP HFLS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 65s 58ms/step - kl_loss: 6.1078 - loss: 13.7197 - reconstruction_loss: 10.6658 - val_kl_loss: 6.0945 - val_loss: 13.8505 - val_reconstruction_loss: 10.8033 - learning_rate: 6.2500e-05
# NP TAS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 78s 70ms/step - kl_loss: 6.5544 - loss: 13.0464 - reconstruction_loss: 9.7692 - val_kl_loss: 6.5407 - val_loss: 12.9894 - val_reconstruction_loss: 9.7191 - learning_rate: 6.2500e-05
# NP ZG
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 76s 68ms/step - kl_loss: 12.5878 - loss: 15.9502 - reconstruction_loss: 9.6563 - val_kl_loss: 12.5723 - val_loss: 16.1382 - val_reconstruction_loss: 9.8520 - learning_rate: 1.5625e-05
# NP SIC
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 109s 97ms/step - kl_loss: 15.2074 - loss: 42.8453 - reconstruction_loss: 35.2415 - val_kl_loss: 15.1112 - val_loss: 46.5071 - val_reconstruction_loss: 38.9515 - learning_rate: 1.0000e-05




In [ ]:
# NP HFLS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 99s 89ms/step - kl_loss: 4.4266 - loss: 17.2497 - reconstruction_loss: 12.8231 - val_kl_loss: 4.3835 - val_loss: 17.3336 - val_reconstruction_loss: 12.9501 - learning_rate: 1.0000e-04
# NP TAS
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 101s 90ms/step - kl_loss: 5.0136 - loss: 17.2999 - reconstruction_loss: 12.2863 - val_kl_loss: 4.9997 - val_loss: 17.1716 - val_reconstruction_loss: 12.1719 - learning_rate: 1.0000e-04
# NP ZG
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 104s 93ms/step - kl_loss: 9.3876 - loss: 22.5601 - reconstruction_loss: 13.1725 - val_kl_loss: 9.4041 - val_loss: 22.6035 - val_reconstruction_loss: 13.1994 - learning_rate: 1.0000e-04
# NP SIC zero fill
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 154s 117ms/step - kl_loss: 11.5868 - loss: 55.5626 - reconstruction_loss: 43.9759 - val_kl_loss: 11.5545 - val_loss: 57.6794 - val_reconstruction_loss: 46.1249 - learning_rate: 1.0000e-04
# NP SIC -1   fill
# 1121/1121 ━━━━━━━━━━━━━━━━━━━━ 130s 116ms/step - kl_loss: 11.7088 - loss: 56.2928 - reconstruction_loss: 44.5840 - val_kl_loss: 11.6394 - val_loss: 57.8884 - val_reconstruction_loss: 46.2490 - learning_rate: 1.0000e-04


In [ ]:
def load_and_clean_data(filepath, memmap_path):
    ds = xr.open_dataset(filepath, engine="h5netcdf", chunks="auto")

    if "TAS" in filepath:
        combined = ds['tas_lr'].fillna(ds['tas_sr']).dropna(dim="lat", how="any").rename("tas")
    elif "hfls" in filepath:
        combined = ds['hfls_lr'].fillna(ds['hfls_sr']).dropna(dim="lat", how="any").rename("hfls")
    elif "zg" in filepath:
        da_lr = ds['zg_lr'].rename({'member_lr': 'member'})
        da_sr = ds['zg_sr'].rename({'member_sr': 'member'})
        zg_combined = xr.concat([da_lr, da_sr], dim='member')
        combined = zg_combined.squeeze('plev').dropna(dim="lat", how="any").rename("hfls")
        del da_lr, da_sr, zg_combined
    elif "NP_SIC" in filepath:
        combined = ds['__xarray_dataarray_variable__'].dropna(dim="lat", how="any").rename("sic")
        combined = combined.pad(lat=(0, 2), constant_values=-1.0)

    elif "BK_SIC" in filepath:
        combined = ds['__xarray_dataarray_variable__'].dropna(dim="lat", how="any").rename("sic")
        combined = combined.pad(lon=(0, 13), constant_values=-1.0)
    else:
        var_name = [v for v in ds.data_vars if v not in ['lat', 'lon', 'time', 'member']][0]
        combined = ds[var_name].dropna(dim="lat", how="any")

    combined_stacked = combined.stack(samples=("member", "time")).transpose("samples", "lat", "lon")

    members = combined_stacked['member'].values
    times = combined_stacked['time'].values

    total_samples = len(members)
    lat_dim = combined_stacked.shape[1]
    lon_dim = combined_stacked.shape[2]
    memmap_shape = (total_samples, lat_dim, lon_dim, 1)

    print(f"Allocating {memmap_shape} memory-map on disk...")
    X_memmap = np.memmap(memmap_path, dtype='float32', mode='w+', shape=memmap_shape)

    chunk_size = 2000
    for i in range(0, total_samples, chunk_size):
        end = min(i + chunk_size, total_samples)
        chunk = combined_stacked.isel(samples=slice(i, end)).values.astype(np.float32)
        X_memmap[i:end, :, :, 0] = chunk

    X_memmap.flush()

    del ds, combined, combined_stacked
    if 'chunk' in locals(): del chunk
    gc.collect()

    X_train = np.memmap(memmap_path, dtype='float32', mode='r', shape=memmap_shape)

    return X_train, members, times

DRIVE_PATH = '/content/drive/MyDrive/python/Masters data'

file_config = {
    "BK_TAS_full.nc": build_BK_vae,
    "BK_hfls_full.nc": build_BK_vae,
    "BK_zg_full.nc": build_BK_vae,
    "NAO_hfls_full.nc": build_NAO_vae,
    "NAO_TAS_full.nc": build_NAO_vae,
    "NAO_zg_full.nc": build_NAO_vae,
    "NP_hfls_full.nc": build_NP_vae,
    "NP_TAS_full.nc": build_NP_vae,
    "NP_zg_full.nc": build_NP_vae,
    "NP_SIC_full.nc": build_NP_SIC_vae,
    "BK_SIC_full.nc": build_BK_SIC_vae
}

In [ ]:
evaluation_results = []

for filename, arch_builder in file_config.items():
    checkpoint_path = os.path.join(DRIVE_PATH, filename.replace(".nc", "_checkpoint_ae.weights.h5"))

    source_nc_path = os.path.join(DRIVE_PATH, filename)
    local_memmap_path = f"/content/{filename.replace('.nc', '_memmap.dat')}"

    if not os.path.exists(checkpoint_path):
        print(f"Skipping {filename}: No saved weights found.")
        continue

    print(f"\n{'='*60}\nEvaluating: {filename}\n{'='*60}")

    try:
        X_train, _, _ = load_and_clean_data(source_nc_path, local_memmap_path)

        split_idx = int(len(X_train) * 0.8)
        X_val_split = X_train[split_idx:]

        def make_memmap_gen(memmap_arr):
            def _gen():
                for i in range(len(memmap_arr)):
                    yield (memmap_arr[i], memmap_arr[i])
            return _gen

        lat_dim, lon_dim = X_train.shape[1], X_train.shape[2]
        spatial_shape = (lat_dim, lon_dim, 1)
        output_sig = (
            tf.TensorSpec(shape=spatial_shape, dtype=tf.float32),
            tf.TensorSpec(shape=spatial_shape, dtype=tf.float32)
        )

        with tf.device('/CPU:0'):
            val_ds = tf.data.Dataset.from_generator(
                make_memmap_gen(X_val_split), output_signature=output_sig
            ).batch(64).prefetch(tf.data.AUTOTUNE)

        encoder, decoder = arch_builder()
        vae = VAE(encoder, decoder)
        vae.build((None, lat_dim, lon_dim, 1))
        vae.compile(optimizer=tf.keras.optimizers.Adam())

        print(f"Loading best weights from {checkpoint_path}...")
        vae.load_weights(checkpoint_path)

        print("Running validation pass...")
        metrics = vae.evaluate(val_ds, return_dict=True)

        evaluation_results.append({
            "Dataset": filename,
            "Total Val Loss": round(float(metrics['loss']), 4),
            "Val Reconstruction Loss": round(float(metrics['reconstruction_loss']), 4),
            "Val KL Loss": round(float(metrics['kl_loss']), 4)
        })

    except Exception as e:
        print(f"Failed to evaluate {filename}: {e}")

    finally:
        if os.path.exists(local_memmap_path): os.remove(local_memmap_path)
        tf.keras.backend.clear_session()

print("\n\n=== FINAL EVALUATION TABLE ===")
results_df = pd.DataFrame(evaluation_results)
print(results_df.to_string(index=False))


results_df.to_csv(os.path.join(DRIVE_PATH, "VAE_Evaluation_Metrics.csv"), index=False)


Evaluating: BK_TAS_full.nc
Allocating (89604, 24, 72, 1) memory-map on disk...
Loading best weights from /content/drive/MyDrive/python/Masters data/BK_TAS_full_checkpoint_ae.weights.h5...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 54 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Running validation pass...
281/281 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - kl_loss: 515.0189 - loss: 0.0652 - reconstruction_loss: 0.0652


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Evaluating: BK_hfls_full.nc
Allocating (89604, 24, 72, 1) memory-map on disk...
Loading best weights from /content/drive/MyDrive/python/Masters data/BK_hfls_full_checkpoint_ae.weights.h5...


...

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 54 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Running validation pass...
281/281 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - kl_loss: 462.9591 - loss: 0.0069 - reconstruction_loss: 0.0069


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Evaluating: NAO_hfls_full.nc
Allocating (89604, 40, 40, 1) memory-map on disk...


KeyboardInterrupt: 

Encoder: 4357280
Decoder: 4363265
Total: 8720545


In [ ]:
# BK
# Encoder: 7.240.864
# Decoder: 7.252.481
# Total:   14.493.345

# BK SIC
# Encoder: 4.357.280
# Decoder: 4.363.265
# Total:   8.720.545

# NAO
# Encoder: 3.734.944
# Decoder: 3.739.009
# Total:   7.473.953

# NP
# Encoder: 51.901.952
# Decoder: 51.860.481
# Total:   103.762.433

# NP SIC
# Encoder: 95.942.144
# Decoder: 95.922.177
# Total:   191.864.321